In [ ]:
# ==========================================
# 🏥 MOBILE DOC - HIGH PERFORMANCE BACKEND
# ==========================================
!pip install -q -U "transformers>=4.48.0" accelerate bitsandbytes "fastapi>=0.115.0" uvicorn pyngrok python-multipart nest_asyncio

import os
import io
import torch
import uvicorn
import nest_asyncio
import traceback
from PIL import Image
from fastapi import FastAPI, UploadFile, File, Form
from pydantic import BaseModel
from contextlib import asynccontextmanager
from pyngrok import ngrok
from transformers import (
    AutoProcessor, 
    AutoModelForImageTextToText, 
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

# --- CONFIGURATION ---

from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
try:
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    NGROK_TOKEN = user_secrets.get_secret("NGROK_TOKEN")
    print("✅ Successfully loaded API Keys from Kaggle Secrets.")
except Exception as e:
    print("⚠️ Keys not found! Please add 'HF_TOKEN' and 'NGROK_TOKEN' in Add-ons > Secrets.")
STATIC_DOMAIN = "choice-peacock-presently.ngrok-free.app"

# 🚀 MODEL IDs
CHAT_ID = "google/gemma-2-2b-it" 
VISION_ID = "google/medgemma-1.5-4b-it" 

# --- DOCTOR LOGIC ---
DOCTOR_INSTRUCTIONS = """
You are **Dr. Mobile Doc**, a smart Nigerian Medical Consultant.

### 🛑 CRITICAL CONVERSATION RULES:
1. **CHECK HISTORY FIRST:** Look at the [RECENT CHAT HISTORY].
   * If you have already greeted the patient (Sannu/Hello), **DO NOT GREET AGAIN**.
   * If you just asked a question (e.g., "How long?"), and the user replies (e.g., "4 hours"), **ACCEPT THE ANSWER**. Do not repeat the question.
2. **DIRECT ANSWERS:** If the user answers a question, immediately provide the medical advice or next step.
   * *Bad:* "How long have you been fasting?" (Repeated)
   * *Good:* "Okay, 4 hours is a while. Since you have an ulcer, please break your fast gently with Pap, not spicy food."
3. **HAUSA CONTEXT:** Understand "lafiya" (fine), "azumi" (fasting), "ciwon" (pain).
4. **NO REPETITION:** Never ask the same question twice in a row.

### 🏥 MEDICAL BEHAVIOR:
* **INTENT:** Analyze if the user is answering a question, asking for advice, or reporting a new symptom.
* **CONTEXT:** Use the [SYSTEM CONTEXT] (Ulcer, Hypertension) to tailor advice, but prioritize the **Current Conversation Flow**.
* **CALORIES:** If asked about food, give a rough Nigerian estimate (e.g., "Fried yam ~400kcal") and safety verdict.

### 🚨 TRIAGE:
* Chest Pain/Breathing Issues = "🚨 **EMERGENCY:** Go to the hospital immediately."
"""
# --- MODEL MANAGEMENT ---
vision_model = None
vision_processor = None
chat_model = None
chat_tokenizer = None

@asynccontextmanager
async def lifespan(app: FastAPI):
    global vision_model, vision_processor, chat_model, chat_tokenizer
    
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    # 🚀 BOTH MODELS LOAD DIRECTLY TO GPU (No CPU offloading needed!)
    print("\n⏳ LOADING CLINICAL BRAIN (Gemma 2 2B)...")
    chat_tokenizer = AutoTokenizer.from_pretrained(CHAT_ID, token=HF_TOKEN)
    chat_tokenizer.pad_token = chat_tokenizer.eos_token 
    chat_model = AutoModelForCausalLM.from_pretrained(
        CHAT_ID, quantization_config=bnb_config, device_map="cuda:0", token=HF_TOKEN
    )

    print("\n⏳ LOADING MEDICAL EYES (MedGemma 1.5 4B)...")
    vision_processor = AutoProcessor.from_pretrained(VISION_ID, token=HF_TOKEN)
    vision_model = AutoModelForImageTextToText.from_pretrained(
        VISION_ID, quantization_config=bnb_config, device_map="cuda:0", token=HF_TOKEN
    )
    
    print("✅ BOTH MODELS LOADED ON GPU SUCCESSFULLY")
    yield
    del vision_model, chat_model
    torch.cuda.empty_cache()

app = FastAPI(lifespan=lifespan)
nest_asyncio.apply()

# --- ENDPOINTS ---
class ChatReq(BaseModel):
    user_text: str

@app.post("/chat")
async def chat_endpoint(req: ChatReq):
    if not chat_model: return {"response": "System Error: Brain not loaded."}

    try:
        # INTELLIGENT PARSING
        latest_msg = req.user_text
        if "[CURRENT MESSAGE]" in req.user_text:
            latest_msg = req.user_text.split("[CURRENT MESSAGE]")[-1].strip()
        elif "USER SAYS:" in req.user_text:
            latest_msg = req.user_text.split("USER SAYS:")[-1].strip()
            
        print(f"\n{'='*50}\n🗣️ DEBUG - CONTEXT AWARE INPUT:\n👉 LATEST: {latest_msg}\n{'='*50}")

        messages = [{"role": "user", "content": f"{DOCTOR_INSTRUCTIONS}\n\n{req.user_text}"}]
        prompt_text = chat_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = chat_tokenizer(prompt_text, return_tensors="pt", padding=True, truncation=True).to("cuda:0")

        # Lower temperature slightly to 0.4 to stop it from wandering/hallucinating greetings
        outputs = chat_model.generate(**inputs, max_new_tokens=250, temperature=0.4, do_sample=True, pad_token_id=chat_tokenizer.eos_token_id)
        response = chat_tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
        
        print(f"\n🩺 DEBUG - AI REPLY:\n{response.strip()}\n{'='*50}")

        suggested_tests = []
        if "MP" in response or "Malaria" in response: suggested_tests.append("Malaria Parasite (MP) Test")
        if "Widal" in response or "Salmonella" in response: suggested_tests.append("Widal Reaction Test")

        return {
            "response": response.strip(),
            "suggested_tests": suggested_tests
        }
    except Exception as e:
        traceback.print_exc()
        return {"response": f"Chat Error: {str(e)}"}

@app.post("/analyze_image")
async def analyze_endpoint(
    file: UploadFile = File(...), 
    description: str = Form(""), 
    mode: str = Form("general")
):
    if not vision_model: return {"response": "System Error: Eyes not loaded."}
    
    try:
        image_bytes = await file.read()
        image = Image.open(io.BytesIO(image_bytes)).convert("RGB").resize((448, 448)) 
        raw_input = description
        food_name = raw_input.split('(')[0].strip() if '(' in raw_input else raw_input

        print(f"\n{'='*50}\n📸 DEBUG - VISION CONTEXT:\nMode: {mode}\nText: {raw_input}\n{'='*50}")

        if mode == "diet":
            task_prompt = (
                f"The user describes this food as: {food_name}. "
                f"Context from App: {raw_input}. "  
                "Do not sound like an AI, Do not Hallucinate."
                "You are a professional and certified Nigerian Medical Nutritionist Analyze the diet: "                
                "1. **Identification & Macros:** What is this? (Carbs/Protein/Fat). Calories? "
                "2. **Health Check:** Look at the 'Context' above. Does the patient have Diabetes, Ulcer, or Pregnancy? Is this safe? "
                "3. **Meal Timing:** check 'Last Meal' in context. Is this good timing? "
                "4. **Verdict:** Suggest a portion size (e.g., '1 wrap')."
                "5. **Recommendation:** "
                "   - If SAFE: Suggest a portion size. "
                "   - If UNSAFE: Suggest a LIGHTER Nigerian alternative (e.g., 'Try Pap/Akamu', 'Pepper Soup', 'Moi-moi'). "
                "   - **IMPORTANT:** Do NOT recommend starvation. Always suggest hydration or a light meal."
            )
        elif mode == "lab":
            task_prompt = (
                "Structured Lab Data Extraction. Identify abnormal biomarkers and main diagnosis. "
                f"Context: {raw_input}."
            )
        elif mode == "scan":
            task_prompt = (
                "Radiological Interpretation. Identify pathologies or irregularities. "
                f"Context: {raw_input}. Provide a professional impression."
            )
        else:
            task_prompt = f"General medical analysis. Context: {raw_input}."

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": task_prompt}
                ]
            }
        ]

        inputs = vision_processor.apply_chat_template(
            messages, 
            add_generation_prompt=True, 
            tokenize=True, 
            return_dict=True, 
            return_tensors="pt"
        ).to("cuda:0", dtype=torch.float16)

        input_len = inputs["input_ids"].shape[-1]
        
        outputs = vision_model.generate(
            **inputs, 
            max_new_tokens=400, 
            do_sample=False,
            num_beams=2
        )
        
        response = vision_processor.decode(outputs[0][input_len:], skip_special_tokens=True)
        print(f"\n{'='*50}\n👁️ DEBUG - VISION REPLY:\n{response.strip()}\n{'='*50}")

        return {"response": response.strip()}

    except Exception as e:
        traceback.print_exc()
        return {"response": f"Vision System Error: {str(e)}"}

# --- SERVER STARTUP ---
if __name__ == "__main__":
    ngrok.set_auth_token(NGROK_TOKEN)
    try:
        public_url = ngrok.connect(8000, domain=STATIC_DOMAIN).public_url
        print(f"\n🚀 BACKEND LIVE: {public_url}")
    except:
        public_url = ngrok.connect(8000).public_url
        print(f"🚀 SERVER LIVE AT: {public_url}")

    config = uvicorn.Config(app, host="0.0.0.0", port=8000)
    server = uvicorn.Server(config)
    await server.serve()